# Instituto Tecnológico y de Estuidos Superiores de Monterrey Campus Querétaro
## Arturo Sánchez Rodríguez | A01275427
# OnePiece


In [ ]:
#Importar las librerías que vamos a utilizar en el proyecto de clasificación de imágenes
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras import models, layers

In [ ]:

onePiece_data = "OnePiece/Data/Data" #Definimos la ruta de nuestro dataset para luego poder llamarlo
print(os.listdir("OnePiece/Data/Data/")[::]) #Vemos las clases que tenemos en nuestro dataset


## Aqui lo que tenemos que hacer es convertir las imágenes a números para que la computadora las pueda leer

In [ ]:
#A diferencia con el dataset de MNIST que ya tiene las imágenes convertidas en números y listas para ser utilizadas, con nuestro modelo no esta asi entonces como la computadora no puede ver las imágenes tenemos que convertirlos en números.

def cargar_img(path, img_size=(64, 64)):
    images = []
    labels = []

    class_names = sorted([
        clase for clase in os.listdir(path)
        if os.path.isdir(os.path.join(path, clase))
    ])

    for label_index, class_name in enumerate(class_names):
        class_path = os.path.join(path, class_name)
        for file_name in os.listdir(class_path):
            if file_name.lower().endswith((".png", ".jpg", "jpeg")):
                img_path = os.path.join(class_path, file_name)
                img = tf.keras.utils.load_img(img_path, target_size=img_size)
                img_array = tf.keras.utils.img_to_array(img)
                images.append(img_array)
                labels.append(label_index)
    images = np.array(images)
    labels = np.array(labels)

    return images, labels, class_names

images, labels, class_names = cargar_img(onePiece_data)


## Imprimimos la forma que tienen las imágenes, sus etiquetas, nombre de las clases y la cantidad de clases que tenemos (Las clases son los personajes en el dataset)

In [ ]:
#Imprimimos la forma de las imágenes, etiquetas, nombres de clases y la cantidad de clases para verificar que todo se haya cargado correctamente
print(images.shape)
print(labels.shape)
print(class_names)
print(len(class_names))

## Separamos el dataset en Test y Train

In [ ]:
#Separamos el dataset en TRAIN y TEST
train_img, test_img, train_labels, test_labels = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train imágenes:", train_img.shape)
print("Test imágenes:", test_img.shape)
print("Train labels:", train_labels.shape)
print("Test labels:", test_labels.shape)

## Escalamos las imágenes para que los valores sean entre 0 y 1


In [ ]:

def scale_img(train_img, test_img):
    train_img = train_img / 255.
    test_img = test_img / 255.
    return train_img, test_img

In [ ]:

train_img, test_img = scale_img(train_img, test_img)

print("train_img.shape:", train_img.shape)
print("test_img.shape:", test_img.shape)
print("train_img.min():", train_img.min(), "train_img.max():", train_img.max())
print("test_img.min():", test_img.min(), "test_img.max():", test_img.max())

## Guardamos las imágenes de test y train en formato .npy para luego pode utilizarlas

In [ ]:


np.save("train_img.npy", train_img)
np.save("test_img.npy", test_img)

np.save("train_labels.npy", train_labels)
np.save("test_labels.npy", test_labels)

print("Se guardaron los test y train en formato .npy de manera correcta!")

In [ ]:
# Visualizar imágenes de ejemplo

import random

plt.figure(figsize=(10,10))

for i in range(9):
    idx = random.randint(0, len(train_img)-1)

    plt.subplot(3,3,i+1)
    plt.imshow(train_img[idx])
    plt.title(class_names[train_labels[idx]])
    plt.axis("off")

plt.show()

# Modelo 1 MLP

In [ ]:
#Tenemos que definir cuántas clases tiene nuestro dataset

num_classes = len(class_names)
print("Número de clases:", num_classes)
print("Clases:", class_names)

In [ ]:
#Creamos el modelo de red neuronal tradicional MLP (Multi-layer Perceptron)

def get_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Flatten(input_shape=input_shape),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = get_model(input_shape=(64, 64, 3), num_classes=num_classes) 
model.summary()

In [ ]:
#Compiamos el modelo 
def compile_model(model):
    model.compile(
        optimizer="adam", #Adam es el algoritmo que modifica los pesos y los ajusta automáticamente
        loss="sparse_categorical_crossentropy", #Función de pérdida para clasificación multiclase
        metrics=["accuracy"] #Métrica para evaluar el rendimiento del modelo
    )

compile_model(model)

In [ ]:
#Entrenamos el modelo
def train_model(model, train_img, train_labels):
    history = model.fit(train_img, train_labels, validation_split=0.2, epochs=25)
    return history

history = train_model(model, train_img, train_labels)

In [ ]:
#Evaluamos el modelo con el set de prueba
test_loss, test_accuracy = model.evaluate(test_img, test_labels)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

# Modelo 2 CNN

In [ ]:
#Construimos el modelo CNN

def get_modelo_CNN(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv2D(6, kernel_size=3, padding="same", activation="relu", input_shape=input_shape),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model 

model_cnn = get_modelo_CNN(input_shape=(64, 64, 3), num_classes=num_classes)
model_cnn.summary()

In [ ]:
#Compilamos y entrenamos el modelo CNN

compile_model(model_cnn)
history_cnn = train_model(model_cnn, train_img, train_labels)

In [ ]:
#Evaluamos el modelo CNN con el set de prueba

test_loss_cnn, test_accuracy_cnn = model_cnn.evaluate(test_img, test_labels)
print("Test loss CNN:", test_loss_cnn)
print("Test accuracy CNN:", test_accuracy_cnn)

# Modelo 3
## CNN + Data Augmentation

In [ ]:
#implementamos data augmentation para mejorar el rendimiento del modelo 2 CNN y evitar overfitting

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    #layers.RandomTranslation(0.1, 0.1),
    #layers.RandomContrast(0.1),
    #layers.RandomFlip("vertical")
])


In [ ]:
#Contruimos el modelo 3 (CNN) con data augmentation

def get_modelo_CNN_aug(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        layers.Conv2D(6, kernel_size=3, padding="same", activation="relu"),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model 

model_cnn_aug = get_modelo_CNN_aug(input_shape=(64, 64, 3), num_classes=num_classes)
model_cnn_aug.summary()


In [ ]:
#Compilamos y entrenamos el modelo CNN con data augmentation

compile_model(model_cnn_aug)
history_cnn_aug = train_model(model_cnn_aug, train_img, train_labels)


In [ ]:
# Evaluamos el modelo CNN con data augmentation con el set de prueba

test_loss_cnn_aug, test_accuracy_cnn_aug = model_cnn_aug.evaluate(test_img, test_labels)
print("Test loss CNN con data augmentation:", test_loss_cnn_aug)
print("Test accuracy CNN con data augmentation:", test_accuracy_cnn_aug)

In [ ]:
# Ver accuracy final de los 3 modelos

#MLP
print(history.history["accuracy"][-1])
print(history.history["val_accuracy"][-1])

#CNN
print(history_cnn.history["accuracy"][-1])
print(history_cnn.history["val_accuracy"][-1])

#CNN con data augmentation
print(history_cnn_aug.history["accuracy"][-1])
print(history_cnn_aug.history["val_accuracy"][-1])
